In [1]:
import pandas as pd

In [2]:
import random

In [3]:
class Player:
    def __init__(self, name):
        self.name = name
        self.hand = []

    def add_card(self, card):
        self.hand.append(card)

    def remove_card(self, card):
        self.hand.remove(card)

    def hand_size(self):
        return len(self.hand)

    def has_won(self):
        return len(self.hand) == 0

    def show_hand(self):
        print(" | ".join(
            f'({i}) {card["value"]} {card["color"]}'
            for i, card in enumerate(self.hand, 1)
        ))

In [4]:
class UnoGame:
    def __init__(self, players):
        self.players = players
        self.deck = []
        self.used = []
        self.current_player = None
        self.last_played = None
        self.normal_order = True
        self.running = True
        self.winner = None


    def create_deck(self):
        self.deck = []
        card_df = pd.read_csv("../data/data.csv")
        for _, row in card_df.iterrows():
            for _ in range(row["count"]):
                self.deck.append({
                    "color": row["color"],
                    "value": row["value"],
                    "effect": row["effect"]
                })


    
    def shuffle_deck(self):
        random.shuffle(self.deck)


    def deal_cards(self):
        for player in self.players:
            for _ in range(7):
                player.hand.append(self.deck.pop())

    def draw_card(self):
        if not self.deck:
            last_card = self.used.pop()
            self.deck = self.used
            random.shuffle(self.deck)
            self.used = [last_card]
        card = self.deck.pop()
        self.current_player.hand.append(card)
    def reset_hand(self):
        for player in self.players:
            player.hand = []


    def legal_cards(self):
        return [
            card for card in self.current_player.hand
            if (
                card["color"] == self.last_played["color"]
                or card["color"] == "Black"
                or card["value"] == self.last_played["value"]
            )
        ]

    def next_player(self):
        if self.current_player is None:
            self.current_player = random.choice(self.players)
            return

        index = self.players.index(self.current_player)

        if self.normal_order:
            self.current_player = self.players[
                (index + 1) % len(self.players)
            ]
        else:
            self.current_player = self.players[
                (index - 1) % len(self.players)
            ]

    def play_turn(self, choice=None):
        if choice is None:
            self.draw_card()
            self.next_player()
            return

        success = self.play_card(choice)

        if success:
            self.next_player()
        else:
            return False


    def check_winner(self):
        if len(self.current_player.hand) == 0:
            self.winner = self.current_player
            self.running = False
            return True

        return False

    def apply_effect(self):
        effect = self.last_played["effect"]

        if effect == "Skip":
            self.next_player()

        elif effect == "Reverse":
            self.normal_order = not self.normal_order

        elif effect == "Draw Two":
            self.next_player()
            self.draw_card()
            self.draw_card()

        elif effect == "Draw Four":
            self.next_player()

            for _ in range(4):
                self.draw_card()


    def play_card(self, choice):
        card = self.current_player.hand[choice - 1]

        if card not in self.legal_cards():
            return False

        played_card = self.current_player.hand.pop(choice - 1)

        self.used.append(played_card)
        self.last_played = played_card

        if len(self.current_player.hand) == 0:
            self.winner = self.current_player
            self.running = False

        return True


    def setup_round(self):
        self.reset_hand()
        self.create_deck()
        self.shuffle_deck()
        self.deal_cards()

        first_card = self.deck.pop()

        self.used.append(first_card)
        self.last_played = first_card

        self.next_player()
            

In [5]:
P1 = Player("P1")
P2 = Player("P2")
P3 = Player("P3")
P4 = Player("P4")

players = [P1, P2, P3, P4]

In [6]:
game = UnoGame(players)

In [7]:
game.setup_round()

In [23]:
game.last_played

{'color': 'Red', 'value': 'Skip', 'effect': 'Skip'}

In [24]:
print("Player turn :", game.current_player.name)
print(game.current_player.show_hand())

Player turn : P4
(1) Draw Two Yellow | (2) 4 Red | (3) 6 Red | (4) 4 Red | (5) 5 Green | (6) 6 Green | (7) Wild Draw Four Black
None


In [16]:
game.play_turn(3)

In [22]:
print(game.last_played["value"] , " | ", game.last_played["color"])

Draw Two  |  Blue


In [16]:
game.current_player.show_hand()

1: 5 Red | 2: 1 Yellow | 3: 8 Yellow | 4: Draw Two Blue | 5: 7 Red | 6: 9 Yellow | 7: 4 Yellow


In [17]:
game.play_turn(4)

In [22]:
game.apply_effect()

In [18]:
game.last_played

{'color': 'Blue', 'value': 'Draw Two', 'effect': 'Draw Two'}